In [0]:
%pip install nemosis

In [0]:
# Connect Databricks to PySpark
from databricks.connect import DatabricksSession

spark = DatabricksSession.builder.getOrCreate()

In [0]:
# Run this cell ONCE to clear all existing widgets
# Then you can delete this cell
try:
    dbutils.widgets.removeAll()
    print("✅ All widgets cleared")
except:
    print("⚠️ No widgets to clear or dbutils not available")

In [0]:
import os
from nemosis import dynamic_data_compiler  # <-- Updated import
import pandas as pd
from datetime import datetime
from pyspark.sql import SparkSession

# --- 1. ENVIRONMENT & AUTHENTICATION ---
# This block detects if you are in VS Code or the Databricks Web UI
# If you are in the Databricks Web UI, it should already has spark and dbutils
try:
    # Check if spark and dbutils exist AND are functional
    spark.sql("SELECT 1").collect()
    dbutils
    print("Environment: Databricks Notebook")
except:
    # If they don't exist or are broken, recreate for the current environment
    try:
        # Try to get the default Databricks Notebook spark session
        spark = SparkSession.builder.getOrCreate()
        from pyspark.dbutils import DBUtils
        dbutils = DBUtils(spark)
        print("Environment: Databricks Notebook (recreated session)")
    except:
        # Fall back to Databricks Connect for VS Code
        from databricks.connect import DatabricksSession
        from databricks.sdk import WorkspaceClient
        
        spark = DatabricksSession.builder.getOrCreate()
        dbutils = WorkspaceClient().dbutils
        print("Environment: VS Code")

# --- 2. PARAMETER INITIALIZATION ---
# Note: When this notebook runs as part of ingest_bronze_job,
# the base_parameters from databricks.yml will override these defaults
dbutils.widgets.text("catalog", "energy_endava_193")
dbutils.widgets.text("schema", "default") 

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

print(f"Using catalog: {catalog}")
print(f"Using schema: {schema}")

# Set the Spark context to your specific sandbox
spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema}")
spark.sql(f"USE SCHEMA {schema}")

print(f"✅ Ingestion Target: {catalog}.{schema}")

# --- 3. INGESTION CONFIGURATION ---
TABLE_NAME = "DISPATCHPRICE"
TARGET_TABLE = "nsw_dispatch_5min"
CACHE_DIR = "/tmp/nemosis_cache"
REGION = "NSW1"

if not os.path.exists(CACHE_DIR):
    os.makedirs(CACHE_DIR)

# 5-Minute Settlement (5MS) officially started Oct 1, 2021
current_year = datetime.now().year # 2026
years_to_process = range(2021, current_year + 1)

# --- 4. DATA PROCESSING LOOP ---
for year in years_to_process:
    # Adjust start date for 2021 to capture the 5MS transition
    start_date = f"{year}/10/01 00:00:00" if year == 2021 else f"{year}/01/01 00:00:00"
    
    # Handle the "till now" requirement for 2026
    if year == current_year:
        end_date = datetime.now().strftime("%Y/%m/%d %H:%M:%S")
    else:
        end_date = f"{year}/12/31 23:55:00"

    print(f"--- ⏳ Downloading: {start_date} to {end_date} ---")
    
    try:
        # Download from AEMO
        df_raw = dynamic_data_compiler(
            start_time=start_date,
            end_time=end_date,
            table_name=TABLE_NAME,
            raw_data_location=CACHE_DIR
        )
        
        # Filter for NSW and convert to Spark
        df_filtered = df_raw[df_raw['REGIONID'] == REGION]
        spark_df = spark.createDataFrame(df_filtered)
        
        # Write to Delta Lake
        # 'append' mode with 'mergeSchema' ensures we don't crash if AEMO adds columns
        if year == 2021:
            spark_df.write.format("delta").mode("overwrite").saveAsTable(TARGET_TABLE)
        else:
            spark_df.write.format("delta").mode("append") \
                .option("mergeSchema", "true") \
                .saveAsTable(TARGET_TABLE)
                
        print(f"✅ Successfully saved {year} data to {TARGET_TABLE}")
        
    except Exception as e:
        print(f"❌ Error in {year}: {e}")

print(f"\n🚀 Mission Accomplished. Data is live at {catalog}.{schema}.{TARGET_TABLE}")